# Midterm Manure Q1 Benchmark

Local VS Code notebook for evaluating ChatbotLP on Midterm Problem 1, Question 1: Manure Management.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

In [ ]:
import os
os.environ["GEMINI_API_KEY"] = ""
os.environ["LLM_PROVIDER"] = "gemini"
os.environ["GEMINI_MODEL"] = "gemini-3-flash-preview"

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

from src.midterm_benchmark import (
    MidtermBenchmarkConfig,
    load_benchmark_files,
    run_midterm_manure_q1_benchmark,
    write_midterm_outputs,
)

## Problem Statement And Reference Solution

In [ ]:
files = load_benchmark_files()
reference = files["reference_solution"]

display(Markdown(files["problem_statement"]))

reference_summary = pd.DataFrame([
    {"metric": "objective_value", "value": reference["objective_value"]},
    {"metric": "demand_revenue", "value": reference["demand_revenue"]},
    {"metric": "transport_cost", "value": reference["transport_cost"]},
    {"metric": "supply_cost", "value": reference["supply_cost"]},
])
display(reference_summary)
display(reference)

## Canonical And Paraphrased Prompt Runs

In [ ]:
USE_LLM = bool(os.environ.get("GEMINI_API_KEY"))

config = MidtermBenchmarkConfig(
    prompt_ids=("canonical", "paraphrased"),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=False,
)
report = run_midterm_manure_q1_benchmark(config=config)

display(pd.DataFrame([report["metadata"]]))
display(report["tables"]["case_summary"])
display(report["tables"]["solve_accuracy"])
display(report["tables"]["interpretation_errors"])
display(report["tables"]["semantic_count_metrics"])
display(report["tables"]["parameter_multiset_metrics"])
display(report["tables"]["topology_metrics"])
display(report["tables"]["route_economics_metrics"])
display(report["tables"]["solver_aggregate_metrics"])
display(report["tables"]["balance_residual_metrics"])
display(report["tables"]["alias_resolution_diagnostics"])
display(report["tables"]["interpretation_metadata"])

In [ ]:
canonical_case = report["cases"][0]
display(canonical_case["validation_result"])
display(canonical_case["solution_checks"]["actual_components"])

## Required Reasoning Prompts

In [ ]:
reasoning_config = MidtermBenchmarkConfig(
    prompt_ids=("canonical",),
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    fallback_to_reference_fixture=True,
    attempt_solve=True,
    run_reasoning=True,
)
reasoning_report = run_midterm_manure_q1_benchmark(config=reasoning_config)
display(reasoning_report["tables"]["reasoning_prompt_success"])

for row in reasoning_report["cases"][0]["reasoning_results"]:
    display(Markdown(f"### {row['prompt_label']}\n\n{row['response_preview']}"))

## Flow Table

In [ ]:
flow_table = pd.DataFrame([
    {"route": "Eau Claire -> Menomonie", "flow_tons": 500},
    {"route": "Eau Claire -> Black River Falls", "flow_tons": 500},
])
display(flow_table)

## Paper-Style Summary Export

In [ ]:
output_dir = REPO_ROOT / "midterm_outputs"
write_midterm_outputs(report, output_dir)
reasoning_report["tables"]["reasoning_prompt_success"].to_csv(
    output_dir / "reasoning_prompt_success_canonical.csv",
    index=False,
)
flow_table.to_csv(output_dir / "manure_q1_flow_table.csv", index=False)

paper_summary = report["tables"]["case_summary"][[
    "prompt_id",
    "solver_ready_actual",
    "semantic_structure_pass",
    "solver_aggregate_pass",
    "balance_residual_pass",
    "primary_success",
    "solve_success",
]]
paper_summary.to_csv(output_dir / "paper_style_summary.csv", index=False)
display(paper_summary)
output_dir